# Week 2 — Topic Modeling & Department Categorization
## AI-Driven Citizen Grievance & Sentiment Analysis System

**Objective:** Convert cleaned text into numerical vectors using TF-IDF and train supervised classification models to automatically route complaints to the correct government department.

**Roadmap for this notebook:**
1. Load preprocessed data from Week 1
2. Feature Engineering — TF-IDF Vectorization
3. Train/Test Split & Label Encoding
4. Baseline Models — Naive Bayes & Logistic Regression
5. Advanced Models — SVM & Random Forest
6. Cross-Validation & Hyperparameter Tuning
7. Model Evaluation & Confusion Matrix
8. Prediction Pipeline Demo
9. Save Models for Week 3

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Sklearn — Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Sklearn — Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Sklearn — Evaluation
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries loaded successfully!')

## Step 2: Load Preprocessed Data from Week 1

> Upload `processed_grievances.csv` (saved at the end of Week 1) to your Colab session, or use the synthetic generator below if it is not available.

In [ ]:
import os

if os.path.exists('processed_grievances.csv'):
    df = pd.read_csv('processed_grievances.csv')
    print(f'✅ Loaded Week 1 data — Shape: {df.shape}')
else:
    print('⚠️  processed_grievances.csv not found. Generating synthetic dataset...')

    import random, re, spacy
    import nltk
    from nltk.corpus import stopwords
    from tqdm import tqdm

    nltk.download('stopwords', quiet=True)
    !python -m spacy download en_core_web_sm -q
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    STOPWORDS = set(stopwords.words('english'))

    random.seed(42)
    templates = {
        'Water Supply':    [
            'Water supply is completely cut off in our area for {n} days.',
            'The tap water has a foul smell and brown color, undrinkable.',
            'Low water pressure makes it impossible to shower or cook.',
            'No running water since {n} days. Residents are struggling.',
            'Water leakage from the main pipeline is wasting thousands of liters.',
            'Contaminated water supply is causing illness in the neighborhood.',
            'Water connection has been disconnected without prior notice.',
            'Water tanker has not arrived for {n} days despite repeated requests.',
        ],
        'Electricity':     [
            'Power outage for {n} hours with no communication from the utility.',
            'Frequent voltage fluctuations are damaging home appliances.',
            'Street lights in sector {n} have not been working for weeks.',
            'Electric pole is dangerously tilted and could collapse any time.',
            'Electricity bill is extremely high despite low consumption this month.',
            'Transformer has blown in our colony. No power since {n} days.',
            'Live wire is exposed near the playground, risk to children.',
            'Power cut during exam season is affecting students badly.',
        ],
        'Roads & Transport':[
            'Massive potholes on main road have damaged several vehicles.',
            'The road has not been repaired despite {n} complaints filed.',
            'No footpath available. Pedestrians are forced to walk on the road.',
            'Bus route {n} has been cancelled without any announcement.',
            'Road digging completed but surface is not restored. Dangerous for bikes.',
            'Traffic signal at the main junction is not functioning.',
            'Overflowing drain is blocking the road and causing waterlogging.',
            'No speed breaker near the school zone. Accidents are frequent.',
        ],
        'Sanitation':      [
            'Garbage has not been collected for {n} days. Stench is unbearable.',
            'Open drain next to residential area is a major health hazard.',
            'Sanitation workers are not cleaning the public toilets regularly.',
            'Waste dumped near park is attracting rats and stray animals.',
            'Sewage overflow on the main street. Urgent action required.',
            'Community dustbin is overflowing and spreading disease.',
            'Dead animals not removed from the street for {n} days.',
            'Illegal dumping of construction debris on public land.',
        ],
        'Public Safety':   [
            'Street lights are not working. Area is unsafe at night.',
            'Frequent theft incidents in sector {n}. Need police patrol.',
            'Stray dogs are attacking children near the school.',
            'Loud noise and anti-social activity near residential colony.',
            'Illegal construction blocking the emergency access road.',
            'No CCTV cameras installed despite repeated demands.',
            'Drug peddling activity openly happening near the park.',
            'Speeding vehicles are a daily danger on the inner road.',
        ],
    }

    rows = []
    for dept, phrases in templates.items():
        for _ in range(250):
            phrase = random.choice(phrases).format(n=random.randint(2, 15))
            rows.append({'narrative': phrase, 'product': dept})

    df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_text'] = df['narrative'].apply(clean_text)

    processed = []
    for doc in tqdm(nlp.pipe(df['cleaned_text'], batch_size=500), total=len(df)):
        tokens = [t.lemma_ for t in doc if t.text not in STOPWORDS and len(t.text) > 2]
        processed.append(' '.join(tokens))
    df['processed_text'] = processed

    df.to_csv('processed_grievances.csv', index=False)
    print(f'✅ Synthetic dataset generated and saved — Shape: {df.shape}')

print(f'\nColumns: {list(df.columns)}')
print(f'Departments: {df["product"].unique()}')
print(f'Class distribution:\n{df["product"].value_counts()}')

## Step 3: Feature Engineering — TF-IDF Vectorization

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts raw text into a numerical matrix where each word gets a weight based on:
- **TF** — how often it appears in a document
- **IDF** — how rare it is across all documents (penalizes common words)

We experiment with both **unigrams** and **bigrams** to capture phrases like `"water supply"` or `"power outage"`.

In [ ]:
# Encode target labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['product'])

print('Label Encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

X = df['processed_text']
y = df['label']

In [ ]:
# Train/Test Split — Stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train class distribution:\n{pd.Series(y_train).value_counts().sort_index()}')

In [ ]:
# TF-IDF Vectorizer — Unigrams + Bigrams
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # capture unigrams and bigrams
    max_features=10000,   # top 10k features
    sublinear_tf=True,    # apply log normalization to TF
    min_df=2,             # ignore terms appearing in fewer than 2 docs
    strip_accents='unicode',
    analyzer='word'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')

In [ ]:
# Visualize top TF-IDF features per department
feature_names = tfidf.get_feature_names_out()

fig, axes = plt.subplots(1, len(le.classes_), figsize=(20, 5))
colors = sns.color_palette('Set2', len(le.classes_))

for i, (cls, ax, color) in enumerate(zip(le.classes_, axes, colors)):
    # Get indices of samples belonging to this class in training set
    indices = [j for j, label in enumerate(y_train) if label == i]
    if not indices:
        continue
    # Sum TF-IDF weights for this class
    class_tfidf = X_train_tfidf[indices].sum(axis=0).A1
    top_idx = class_tfidf.argsort()[-10:][::-1]
    top_words = [feature_names[k] for k in top_idx]
    top_scores = [class_tfidf[k] for k in top_idx]

    ax.barh(top_words[::-1], top_scores[::-1], color=color, edgecolor='white')
    ax.set_title(f'{cls}', fontsize=10, fontweight='bold')
    ax.set_xlabel('TF-IDF Weight')

plt.suptitle('Top 10 TF-IDF Keywords per Department', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tfidf_top_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: tfidf_top_features.png')

## Step 4: Baseline Models

We start with fast, interpretable baselines before moving to more complex models.

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, label_names):
    """Train, predict, and return a results dictionary."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc   = accuracy_score(y_te, y_pred)
    macro = f1_score(y_te, y_pred, average='macro')
    weighted = f1_score(y_te, y_pred, average='weighted')

    print(f'\n{'='*55}')
    print(f'  Model: {model_name}')
    print(f'  Accuracy        : {acc:.4f}')
    print(f'  Macro F1-Score  : {macro:.4f}')
    print(f'  Weighted F1     : {weighted:.4f}')
    print(f'{'='*55}')
    print(classification_report(y_te, y_pred, target_names=label_names))

    return {
        'model_name': model_name,
        'model_obj': model,
        'y_pred': y_pred,
        'accuracy': acc,
        'macro_f1': macro,
        'weighted_f1': weighted
    }

LABELS = le.classes_.tolist()
results = []

In [ ]:
# --- Baseline 1: Multinomial Naive Bayes ---
nb = MultinomialNB(alpha=0.5)
res_nb = evaluate_model(nb, X_train_tfidf, y_train, X_test_tfidf, y_test,
                         'Naive Bayes', LABELS)
results.append(res_nb)

In [ ]:
# --- Baseline 2: Logistic Regression ---
lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                         multi_class='multinomial', random_state=42)
res_lr = evaluate_model(lr, X_train_tfidf, y_train, X_test_tfidf, y_test,
                         'Logistic Regression', LABELS)
results.append(res_lr)

## Step 5: Advanced Models

In [ ]:
# --- Advanced 1: Linear SVM ---
# LinearSVC is extremely efficient on sparse TF-IDF matrices
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
res_svm = evaluate_model(svm, X_train_tfidf, y_train, X_test_tfidf, y_test,
                          'Linear SVM', LABELS)
results.append(res_svm)

In [ ]:
# --- Advanced 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                              min_samples_leaf=2, random_state=42, n_jobs=-1)
res_rf = evaluate_model(rf, X_train_tfidf, y_train, X_test_tfidf, y_test,
                         'Random Forest', LABELS)
results.append(res_rf)

## Step 6: Cross-Validation

Cross-validation ensures the model generalizes well and we aren't overfitting to a single train/test split. We use **Stratified 5-Fold CV** which preserves class proportions in each fold.

In [ ]:
# Full TF-IDF on ALL data for CV (refitting from scratch)
tfidf_cv = TfidfVectorizer(ngram_range=(1,2), max_features=10000,
                            sublinear_tf=True, min_df=2)
X_all_tfidf = tfidf_cv.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Naive Bayes'         : MultinomialNB(alpha=0.5),
    'Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                                               multi_class='multinomial', random_state=42),
    'Linear SVM'          : LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

cv_results = {}
print('Running 5-Fold Stratified Cross-Validation...\n')

for name, model in cv_models.items():
    scores = cross_val_score(model, X_all_tfidf, y,
                              cv=skf, scoring='f1_macro', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s} | Mean Macro F1: {scores.mean():.4f} ± {scores.std():.4f}')

print('\n✅ Cross-Validation complete!')

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 5))

names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds  = [cv_results[n].std() for n in names]
colors = sns.color_palette('Set2', len(names))

bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=colors, edgecolor='white', error_kw={'elinewidth': 2})

# Annotate bars
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.set_ylabel('Macro F1-Score', fontsize=12)
ax.set_title('5-Fold Cross-Validation — Macro F1 by Model\n(error bars = ±1 std dev)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=11)

plt.tight_layout()
plt.savefig('cv_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: cv_model_comparison.png')

## Step 7: Hyperparameter Tuning for Best Model

We tune the best-performing model (typically SVM or Logistic Regression) using **GridSearchCV**.

In [ ]:
# Build a full Pipeline: TF-IDF + SVM  (tuning both steps together)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, min_df=2, analyzer='word')),
    ('clf', LinearSVC(max_iter=2000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_features': [5000, 10000],
    'clf__C': [0.1, 1.0, 5.0],
}

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\n✅ Best Parameters : {grid_search.best_params_}')
print(f'   Best CV Macro F1: {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate the tuned best model on the held-out test set
best_pipeline = grid_search.best_estimator_
y_pred_best = best_pipeline.predict(X_test)

print('\n===  Tuned SVM Pipeline — Test Set Performance  ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Macro F1  : {f1_score(y_test, y_pred_best, average="macro"):.4f}')
print(f'Weighted F1: {f1_score(y_test, y_pred_best, average="weighted"):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=LABELS))

## Step 8: Confusion Matrix & Model Comparison

In [ ]:
# Confusion Matrix for the best model
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=30)
ax.set_title('Confusion Matrix — Tuned SVM Pipeline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_best_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrix_best_model.png')

In [ ]:
# Summary table comparing all models
summary_df = pd.DataFrame([
    {
        'Model'       : r['model_name'],
        'Accuracy'    : round(r['accuracy'], 4),
        'Macro F1'    : round(r['macro_f1'], 4),
        'Weighted F1' : round(r['weighted_f1'], 4),
    }
    for r in results
] + [{
    'Model'       : 'Tuned SVM Pipeline',
    'Accuracy'    : round(accuracy_score(y_test, y_pred_best), 4),
    'Macro F1'    : round(f1_score(y_test, y_pred_best, average='macro'), 4),
    'Weighted F1' : round(f1_score(y_test, y_pred_best, average='weighted'), 4),
}])

summary_df = summary_df.sort_values('Macro F1', ascending=False).reset_index(drop=True)
print('\n📊 Model Comparison Summary:')
print(summary_df.to_string(index=False))

In [ ]:
# Grouped bar chart — Accuracy vs Macro F1 vs Weighted F1
x = np.arange(len(summary_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width, summary_df['Accuracy'],    width, label='Accuracy',    color='#4e9af1', edgecolor='white')
b2 = ax.bar(x,          summary_df['Macro F1'],   width, label='Macro F1',    color='#f1c84e', edgecolor='white')
b3 = ax.bar(x + width,  summary_df['Weighted F1'],width, label='Weighted F1', color='#6fd16f', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(summary_df['Model'], rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: model_comparison_bar.png')

## Step 9: Prediction Pipeline Demo

Simulating how the system routes a raw citizen complaint to the correct department in real time.

In [ ]:
import re

def preprocess_single(text):
    """Minimal preprocessing for inference (no spaCy required)."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def predict_department(complaint_text, pipeline, label_encoder):
    """Route a raw complaint to its department."""
    cleaned = preprocess_single(complaint_text)
    pred_label = pipeline.predict([cleaned])[0]
    department = label_encoder.inverse_transform([pred_label])[0]
    return department

# Demo complaints
demo_complaints = [
    'There has been no water in our taps for the last 5 days. Please fix the pipeline immediately!',
    'The entire street has been in darkness for 3 nights because all the street lights are broken.',
    'Huge potholes on the main highway have caused several bike accidents this week.',
    'Garbage bins near the market are overflowing and the smell is absolutely horrible.',
    'There are stray dogs attacking people near the school. Children are terrified.',
    'Our electricity bill is 3 times higher than usual despite using less power this month.',
]

print('🤖 Department Routing Demo\n' + '='*55)
for complaint in demo_complaints:
    dept = predict_department(complaint, best_pipeline, le)
    print(f'📋 Complaint : {complaint[:70]}...')
    print(f'🏛️  Routed to  : {dept}')
    print()

## Step 10: Save Models & Artifacts for Week 3

In [ ]:
# Save best pipeline (TF-IDF + SVM) — used in Week 3 API
joblib.dump(best_pipeline, 'best_department_classifier.pkl')
joblib.dump(le,            'label_encoder.pkl')

# Save summary results
summary_df.to_csv('week2_model_summary.csv', index=False)

print('✅ Saved artifacts for Week 3:')
print('   best_department_classifier.pkl  — Tuned TF-IDF + SVM pipeline')
print('   label_encoder.pkl              — LabelEncoder for decoding predictions')
print('   week2_model_summary.csv        — Model comparison table')
print(f'\nFinal Macro F1 (best model): {f1_score(y_test, y_pred_best, average="macro"):.4f}')

## Summary — Week 2 Deliverables

| Deliverable | Status |
|---|---|
| TF-IDF Vectorization (unigrams + bigrams) | ✅ Complete |
| Per-department top keyword visualization | ✅ Complete |
| Naive Bayes baseline | ✅ Complete |
| Logistic Regression baseline | ✅ Complete |
| Linear SVM | ✅ Complete |
| Random Forest | ✅ Complete |
| Stratified 5-Fold Cross-Validation | ✅ Complete |
| GridSearchCV Hyperparameter Tuning | ✅ Complete |
| Confusion Matrix visualization | ✅ Complete |
| Model comparison bar chart | ✅ Complete |
| Prediction pipeline demo | ✅ Complete |
| Models saved for Week 3 | ✅ Complete |

**Next Steps (Week 3):** Sentiment Analysis — assign urgency scores to complaints using VADER / TextBlob as baselines before fine-tuning a BERT-based model for nuanced emotional tone detection.